## Validation Datasets

Now compare model output to actual station data (similar to comparing ERA5 to weather station data)

In [1]:
#import what you need
import pandas as pd #for station data
import numpy as np #for subtracting arrays
from datetime import datetime, timedelta #to handle the dates
import statistics #to get mean and sd
from math import isnan #get rid of nas
from itertools import filterfalse #get rid of nas
import xarray as xr
from netCDF4 import Dataset #for using wrf.getvar

#to get a list of the files in the folder
import glob 

##attempt to reformat the netcdf in one line
import xwrf

#to quickly calculate relative humidity
from wrf import getvar, interplevel, to_np, latlon_coords, uvmet
import wrf


In [2]:
## read in atmospheric data
d02_2017_names = glob.glob("../../01Data/Houston_MarineHeatWave_2017/wrfout_MarineHeatWave.d02.*")
wrfin = [Dataset(f) for f in d02_2017_names[8:] ] #gives you two days of burn-in time I think
initial_ds = xr.open_dataset(d02_2017_names[0]).xwrf.postprocess() #for land use
T2_data = wrf.getvar(wrfin, "T2", timeidx=wrf.ALL_TIMES)

#ERA5
#era5_sfcMar = xr.load_dataset("../../01Data/SFC/ERA5_WRF_SFC_201703.grib", engine="cfgrib", decode_timedelta=True)
#era5_sfcApr = xr.load_dataset("../../01Data/SFC/ERA5_WRF_SFC_201704.grib", engine="cfgrib", decode_timedelta=True)

#Station data
anglStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/AngletonLakeJackson/GHCNh_USW00012976_2017.psv", sep="|", low_memory=False)
beauStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Beaumont/GHCNh_USW00000313_2017.psv", sep="|", low_memory=False)
conrStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Conroe/GHCNh_USW00053902_2017.psv", sep="|", low_memory=False)
galvStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Galveston/GHCNh_USW00012923_2017.psv", sep="|", low_memory=False)
housStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Houston/GHCNh_USW00012975_2017.psv", sep="|", low_memory=False)

clvlStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Cleveland/GHCNh_USW00000378_2017.psv", sep="|", low_memory=False)
hiolStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/HighIslandOil/GHCNh_USW00000306_2017.psv", sep="|", low_memory=False)
hseaStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/HoustonExecAP/GHCNh_USW00000208_2017.psv", sep="|", low_memory=False)
hsinStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/HoustonIntercontl/GHCNh_USW00012960_2017.psv", sep="|", low_memory=False)
hntsStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Huntsville/GHCNh_USW00053903_2017.psv", sep="|", low_memory=False)
ptarStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/PortArthur/GHCNh_USW00012917_2017.psv", sep="|", low_memory=False)
rbrwStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/RobertRWells/GHCNh_USI0000K66R_2017.psv", sep="|", low_memory=False)
sbolStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/SabineOil/GHCNh_USW00000255_2017.psv", sep="|", low_memory=False)
srgtStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/Sargent/GHCNh_USL000SGNT2_2017.psv", sep="|", low_memory=False)
txptStation = pd.read_csv("../../01Data/ValidationDatasets/WeatherStations/TexasPt/GHCNh_USL000TXPT2_2017.psv", sep="|", low_memory=False)

In [3]:
## Get useful station information for later work

#Station locations
Statlats = [anglStation["LATITUDE"][0], beauStation["LATITUDE"][0], conrStation["LATITUDE"][0], 
        galvStation["LATITUDE"][0], housStation["LATITUDE"][0], clvlStation["LATITUDE"][0], 
        hiolStation["LATITUDE"][0], hseaStation["LATITUDE"][0], hsinStation["LATITUDE"][0], 
        hntsStation["LATITUDE"][0], ptarStation["LATITUDE"][0], rbrwStation["LATITUDE"][0], 
        sbolStation["LATITUDE"][0], srgtStation["LATITUDE"][0], txptStation["LATITUDE"][0]]
Statlons = [anglStation["LONGITUDE"][0], beauStation["LONGITUDE"][0], conrStation["LONGITUDE"][0], 
        galvStation["LONGITUDE"][0], housStation["LONGITUDE"][0], clvlStation["LONGITUDE"][0], 
        hiolStation["LONGITUDE"][0], hseaStation["LONGITUDE"][0], hsinStation["LONGITUDE"][0], 
        hntsStation["LONGITUDE"][0], ptarStation["LONGITUDE"][0], rbrwStation["LONGITUDE"][0], 
        sbolStation["LONGITUDE"][0], srgtStation["LONGITUDE"][0], txptStation["LONGITUDE"][0]]

## Prep some other variables for comparisons
FocalTimePts = [datetime.fromisoformat("2017-03-23T00:00:00"), datetime.fromisoformat("2017-04-01T00:00:00"), 
                datetime.fromisoformat("2017-04-03T00:00:00")]

# Get 2D lat/lon from WRF
lats, lons = wrf.latlon_coords(T2_data)

Start with the weather stations, going in alphabetical order

* Angleton Lake Jackson
* Beaumont
* Conroe
* Galveston
* Houston

Newly added:
* Cleveland
* High Island Oil
* Houston Executive Airport
* Houston Intercontinental Airport
* Huntsville
* Port Arthur
* Robert R Wells
* Sabine Oil
* Sargent
* Texas Point

In [4]:
##get avg temps
def getAvg(df, df2, qc):

    avg_temps = []
    
    #loop through it
    for t in df.Time.values:
    
        focalTime = pd.to_datetime(pd.Timestamp(t))
        # Subset data within the time tolerance
        subset = [
            entry['temperature'] for entry in df2
            if abs(datetime.fromisoformat(entry['DATE']) - focalTime) <= tolerance and str(entry['temperature_Quality_Code']) == qc
        ]
    
        # Compute the average temperature if any entries matched
        if subset:
            avg = sum(subset) / len(subset)
        else:
            avg = float('nan')
            
        avg_temps.append(avg)
        
    return avg_temps

In [5]:
## Comparisons for Angleton Lake Jackson
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[0]) ** 2 + (lons - Statlons[0]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
anglSubset = T2_data[:, j, i]  # time series at nearest grid point

#find corresponding temperature values within some tolerance of the era5 times
# Store average temperatures
avg_temps = []
tolerance= timedelta(minutes=30)
anglStation = anglStation.to_dict('records')

avg_temps = getAvg(anglSubset, anglStation, '5')
    
#get the difference between these time points and the closest station values
anglDiffs = np.array([anglSubset.values - 272.15]).flatten()  - avg_temps
angl_Diffs_nn = list(filterfalse(isnan, anglDiffs))

In [6]:
#get some summary statistics
angl_avgDiff = statistics.mean(angl_Diffs_nn)
angl_medDiff = statistics.median(angl_Diffs_nn)

Beaumont

In [7]:
## Comparisons for Beaumont
# Find subset within the tolerance range
distance = np.sqrt((lats - Statlats[1]) ** 2 + (lons - Statlons[1]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
beauSubset = T2_data[:, j, i]  # time series at nearest grid point

#find corresponding temperature values within some tolerance of the era5 times

# Store average temperatures
avg_temps = []
tolerance= timedelta(minutes=45)
beauStation = beauStation.to_dict('records')

avg_temps = getAvg(beauSubset, beauStation, '1')

#get the difference between these time points and the closest station values
beau_Diffs = np.array([beauSubset.values - 272.15]).flatten() - avg_temps
beau_Diffs_nn = list(filterfalse(isnan, beau_Diffs))
    

In [8]:
#get some summary statistics
beau_avgDiff = statistics.mean(beau_Diffs)
beau_medDiff = statistics.median(beau_Diffs)

Conroe

In [9]:
## Comparisons for Conroe
# Find subset within the tolerance range
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[2]) ** 2 + (lons - Statlons[2]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
conrSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures
avg_temps = []
tolerance= timedelta(minutes=30)
conrStation = conrStation.to_dict('records')

avg_temps = getAvg(conrSubset, conrStation, '5.0')

#get the difference between these time points and the closest station values
conr_Diffs = np.array([conrSubset.values - 272.15]).flatten() - avg_temps
conr_Diffs_nn = list(filterfalse(isnan, conr_Diffs))

In [10]:
#get some summary statistics
conr_avgDiff = statistics.mean(conr_Diffs_nn)
conr_medDiff = statistics.median(conr_Diffs_nn)

Galveston

In [11]:
## Comparisons for Galveston
# Find subset within the tolerance range
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[3]) ** 2 + (lons - Statlons[3]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
galvSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
avg_temps = []
tolerance= timedelta(minutes=30)
galvStation = galvStation.to_dict('records')

avg_temps = getAvg(galvSubset, galvStation, '5.0')
    
#get the difference between these time points and the closest station values
galv_Diffs = np.array([galvSubset.values - 272.15]).flatten()- avg_temps
galv_Diffs_nn = list(filterfalse(isnan, galv_Diffs))

In [12]:
#get some summary statistics
galv_avgDiff = statistics.mean(galv_Diffs_nn)
galv_medDiff = statistics.median(galv_Diffs_nn)

Houston

In [13]:
## Comparisons for Houston
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[4]) ** 2 + (lons - Statlons[4]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
housSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
avg_temps = []
tolerance= timedelta(minutes=30)
housStation = housStation.to_dict('records')

avg_temps = getAvg(housSubset, housStation, '5.0')
    
#get the difference between these time points and the closest station values
hous_Diffs = np.array([housSubset.values - 272.15]).flatten() - avg_temps
hous_Diffs_nn = list(filterfalse(isnan, hous_Diffs))

In [14]:
#get some summary statistics
hous_avgDiff = statistics.mean(hous_Diffs_nn)
hous_medDiff = statistics.median(hous_Diffs_nn)

In [15]:
## Comparisons for Cleveland
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[5]) ** 2 + (lons - Statlons[5]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
clvlSubset = T2_data[:, j, i]  # time series at nearest grid point
# Store average temperatures - March
avg_temps = []
tolerance= timedelta(minutes=30)
clvlStation = clvlStation.to_dict('records')

avg_temps = getAvg(clvlSubset, clvlStation, '1')

#get the difference between these time points and the closest station values
clvl_Diffs = np.array([clvlSubset.values - 272.15]).flatten() - avg_temps
clvl_Diffs_nn = list(filterfalse(isnan, clvl_Diffs))

In [16]:
#get some summary statistics
clvl_avgDiff = statistics.mean(clvl_Diffs_nn)
clvl_medDiff = statistics.median(clvl_Diffs_nn)

In [17]:
## Comparisons for High Island Oil
# Find subset within the tolerance range
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[6]) ** 2 + (lons - Statlons[6]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
hiolSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures 
mar_avg_temps = []
tolerance= timedelta(minutes=30)
hiolStation = hiolStation.to_dict('records')

avg_temps = getAvg(hiolSubset, hiolStation, '1')
    
#get the difference between these time points and the closest station values
hiol_Diffs = np.array([hiolSubset.values - 272.15]).flatten()
hiol_Diffs_nn = list(filterfalse(isnan, hiol_Diffs))

In [18]:
#get some summary statistics
hiol_avgDiff = statistics.mean(hiol_Diffs_nn)
hiol_medDiff = statistics.median(hiol_Diffs_nn)

In [19]:
## Comparisons for Houston Executive Airport
# Find subset within the tolerance range
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[7]) ** 2 + (lons - Statlons[7]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
hseaSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
hseaStation = hseaStation.to_dict('records')

avg_temps = getAvg(hiolSubset, hseaStation, '1')
    
#get the difference between these time points and the closest station values
hsea_Diffs = np.array([hseaSubset.values - 272.15]).flatten() - avg_temps
hsea_Diffs_nn = list(filterfalse(isnan, hsea_Diffs))


In [20]:
#get some summary statistics
hsea_avgDiff = statistics.mean(hsea_Diffs_nn)
hsea_medDiff = statistics.median(hsea_Diffs_nn)

In [21]:
## Comparisons for Houston Intercontinental Airport
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[8]) ** 2 + (lons - Statlons[8]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
hsinSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
avg_temps = []
tolerance= timedelta(minutes=30)
hsinStation = hsinStation.to_dict('records')

avg_temps = getAvg(hsinSubset, hsinStation, '5')
    
#get the difference between these time points and the closest station values
hsin_Diffs = np.array([hsinSubset.values - 272.15]).flatten() - avg_temps
hsin_Diffs_nn = list(filterfalse(isnan, hsin_Diffs))

In [22]:
#get some summary statistics
hsin_avgDiff = statistics.mean(hsin_Diffs_nn)
hsin_medDiff = statistics.median(hsin_Diffs_nn)

In [23]:
## Comparisons for Huntsville
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[9]) ** 2 + (lons - Statlons[9]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
hntsSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
avg_temps = []
tolerance= timedelta(minutes=30)
hntsStation = hntsStation.to_dict('records')

avg_temps = getAvg(hntsSubset, hntsStation, '5')
    
#get the difference between these time points and the closest station values
hnts_Diffs = np.array([hntsSubset.values - 272.15]).flatten() - avg_temps
hnts_Diffs_nn = list(filterfalse(isnan, hnts_Diffs))

In [24]:
#get some summary statistics
hnts_avgDiff = statistics.mean(hnts_Diffs_nn)
hnts_medDiff = statistics.median(hnts_Diffs_nn)

In [25]:
## Comparisons for Port Arthur
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[10]) ** 2 + (lons - Statlons[10]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
ptarSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
avg_temps = []
tolerance= timedelta(minutes=30)
ptarStation = ptarStation.to_dict('records')

avg_temps = getAvg(ptarSubset, ptarStation, '5')
    
#get the difference between these time points and the closest station values
ptar_Diffs = np.array([ptarSubset.values - 272.15]).flatten() - avg_temps
ptar_Diffs_nn = list(filterfalse(isnan, ptar_Diffs))

In [26]:
#get some summary statistics
ptar_avgDiff = statistics.mean(ptar_Diffs_nn)
ptar_medDiff = statistics.median(ptar_Diffs_nn)

In [27]:
## Comparisons for Robert R Wells
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[11]) ** 2 + (lons - Statlons[11]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
rbrwSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
avg_temps = []
tolerance= timedelta(minutes=30)
rbrwStation = rbrwStation.to_dict('records')

avg_temps = getAvg(rbrwSubset, rbrwStation, '1')
    
#get the difference between these time points and the closest station values
rbrw_Diffs = np.array([rbrwSubset.values - 272.15]).flatten() - avg_temps
rbrw_Diffs_nn = list(filterfalse(isnan, rbrw_Diffs))

In [28]:
#get some summary statistics
rbrw_avgDiff = statistics.mean(rbrw_Diffs_nn)
rbrw_medDiff = statistics.median(rbrw_Diffs_nn)

In [29]:
## Comparisons for Sabine Oil
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[12]) ** 2 + (lons - Statlons[12]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
sbolSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures
avg_temps = []
tolerance= timedelta(minutes=30)
sbolStation = sbolStation.to_dict('records')

avg_temps = getAvg(sbolSubset,sbolStation, '5')
    
#get the difference between these time points and the closest station values
sbol_Diffs = np.array([sbolSubset.values - 272.15]).flatten() - avg_temps
sbol_Diffs_nn = list(filterfalse(isnan, sbol_Diffs))

In [30]:
#get some summary statistics
sbol_avgDiff = statistics.mean(sbol_Diffs_nn)
sbol_medDiff = statistics.median(sbol_Diffs_nn)

In [31]:
## Comparisons for Sargent
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[13]) ** 2 + (lons - Statlons[13]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
srgtSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
srgtStation = srgtStation.to_dict('records')

avg_temps = getAvg(srgtSubset, srgtStation, '1.0')
    
#get the difference between these time points and the closest station values
srgt_Diffs = np.array([srgtSubset.values - 272.15]).flatten() - avg_temps
srgt_Diffs_nn = list(filterfalse(isnan, srgt_Diffs))

In [32]:
#get some summary statistics
srgt_avgDiff = statistics.mean(srgt_Diffs_nn)
srgt_medDiff = statistics.median(srgt_Diffs_nn)

In [33]:
## Comparisons for Texas Point
# Compute distance from station to all grid points
distance = np.sqrt((lats - Statlats[14]) ** 2 + (lons - Statlons[14]) ** 2)

# Convert to numpy array to find the indices of the minimum
min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)
j, i = min_dist_idx  # south_north (y), west_east (x)

# Now subset the T2 data at this location
txptSubset = T2_data[:, j, i]  # time series at nearest grid point

# Store average temperatures - March
mar_avg_temps = []
tolerance= timedelta(minutes=30)
txptStation = txptStation.to_dict('records')

avg_temps = getAvg(txptSubset, txptStation, '1.0')
    
#get the difference between these time points and the closest station values
txpt_Diffs = np.array([txptSubset.values - 272.15]).flatten() - avg_temps
txpt_Diffs_nn = list(filterfalse(isnan, txpt_Diffs))

In [34]:
#get some summary statistics
txpt_avgDiff = statistics.mean(txpt_Diffs_nn)
txpt_medDiff = statistics.median(txpt_Diffs_nn)

In [35]:
means = [angl_avgDiff, beau_avgDiff, conr_avgDiff, galv_avgDiff, hous_avgDiff, clvl_avgDiff, 
         hiol_avgDiff, hsea_avgDiff, hsin_avgDiff, hnts_avgDiff, ptar_avgDiff, rbrw_avgDiff, 
         sbol_avgDiff, srgt_avgDiff, txpt_avgDiff]
medians = [angl_medDiff, beau_medDiff, conr_medDiff, galv_medDiff, hous_medDiff, clvl_medDiff, 
         hiol_medDiff, hsea_medDiff, hsin_medDiff, hnts_medDiff, ptar_medDiff, rbrw_medDiff, 
         sbol_medDiff, srgt_medDiff, txpt_medDiff]
OutputDF = pd.DataFrame(np.column_stack((means, medians)), columns=["mean", "median"], 
                        index=["Angleton", "Beaumont", "Conroe", "Galveston", "Houston",
                               "Cleveland", "High Island Oil", "Houston Executive Airport",
                               "Houston Intercontinental Airport", "Huntsville", 
                               "Port Arthur", "Robert R Wells", "Sabine Oil", "Sargent", "Texas Point"])
OutputDF.to_csv("../../03ProcessedData/ModelErrors.csv")

In [46]:
srgtSubset.Time.values

array(['2017-03-25T00:00:00.000000000', '2017-03-25T01:00:00.000000000',
       '2017-03-25T02:00:00.000000000', '2017-03-25T03:00:00.000000000',
       '2017-03-25T04:00:00.000000000', '2017-03-25T05:00:00.000000000',
       '2017-03-25T06:00:00.000000000', '2017-03-25T07:00:00.000000000',
       '2017-03-25T08:00:00.000000000', '2017-03-25T09:00:00.000000000',
       '2017-03-25T10:00:00.000000000', '2017-03-25T11:00:00.000000000',
       '2017-03-25T12:00:00.000000000', '2017-03-25T13:00:00.000000000',
       '2017-03-25T14:00:00.000000000', '2017-03-25T15:00:00.000000000',
       '2017-03-25T16:00:00.000000000', '2017-03-25T17:00:00.000000000',
       '2017-03-25T18:00:00.000000000', '2017-03-25T19:00:00.000000000',
       '2017-03-25T20:00:00.000000000', '2017-03-25T21:00:00.000000000',
       '2017-03-25T22:00:00.000000000', '2017-03-25T23:00:00.000000000',
       '2017-03-26T00:00:00.000000000', '2017-03-26T01:00:00.000000000',
       '2017-03-26T02:00:00.000000000', '2017-03-26